# Multi-Modal Label Coverage Visualisation

This notebook compares the `ocosense`, `eyetracker`, and `empatica` modalities of the shared CLUES dataset (`<CLUES_DIR>/<subject_id>/<modality>/<subject_id>.csv`). Set the `CLUES_DIR` environment variable to the dataset root, or edit `BASE_DATA_DIR` below. It aggregates label distributions per-second and per-sample to help verify synchronisation and coverage before release.

## Setup and Configuration

Import required libraries and configure plotting style.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

try:
    plt.style.use("seaborn-v0_8")
except OSError:
    plt.style.use("ggplot")

## Data Loading Functions

Define utility functions to:
- **Discover subjects**: Automatically find all subject folders in the dataset
- **Normalize labels**: Clean and standardize label names across modalities
- **Load modality data**: Read CSV files for each modality (OCOSense, eye tracker, Empatica)
- **Assign windows**: Segment continuous data into time windows (1-second windows by default)
- **Compute label counts**: Aggregate labels by window or by individual rows

These functions enable consistent processing across all three modalities.

In [ ]:
BASE_DATA_DIR = Path(os.environ.get("CLUES_DIR", "CLUES"))

MODALITY_FOLDERS = {
    "ocosense": "ocosense",
    "eyetracker": "eyetracker",
    "empatica": "empatica",
}

BASE_LABEL_ORDER = [
    "questionnaire",
    "rest",
    "listen_music",
    "passive_viewing",
    "central_position",
    "2_back",
    "3_back",
    "stroop_easy",
    "stroop_difficult",
    "memory_easy",
    "memory_difficult",
    "images_questions_easy",
    "images_questions_difficult",
    "difference_images_easy",
    "difference_images_difficult",
    "pursuit_easy",
    "pursuit_difficult",
]

WINDOW_SIZE = 1.0  # seconds
OVERLAP = 0.0      # seconds

def discover_subjects(base_dir: Path = BASE_DATA_DIR):
    subjects = []
    for path in base_dir.glob("s_*"):
        if not path.is_dir():
            continue
        try:
            subjects.append(int(path.name.split("_")[1]))
        except (IndexError, ValueError):
            continue
    return sorted(subjects)

def normalise_labels(series: pd.Series) -> pd.Series:
    labels = series.fillna("").astype(str).str.strip().str.lower()
    return labels.replace({"": np.nan})

def ensure_label_order(existing_labels) -> list[str]:
    order = [label for label in BASE_LABEL_ORDER if label in existing_labels]
    for label in sorted(existing_labels):
        if label not in order:
            order.append(label)
    return order

def load_modality_data(modality: str, subjects=None, base_dir: Path = BASE_DATA_DIR):
    folder_name = MODALITY_FOLDERS[modality]
    subjects = subjects or discover_subjects(base_dir)
    frames = []
    missing = []
    for subject in tqdm(subjects, desc=f"  Loading subjects", unit="subject"):
        csv_path = base_dir / f"s_{subject:03d}" / folder_name / f"s_{subject:03d}.csv"
        if not csv_path.exists():
            missing.append(csv_path)
            continue
        df = pd.read_csv(csv_path, low_memory=False)
        if "timestamp" not in df.columns:
            raise ValueError(f"'timestamp' column missing in {csv_path}")
        if "label" not in df.columns:
            raise ValueError(f"'label' column missing in {csv_path}")
        df = df.copy()
        df["label"] = normalise_labels(df["label"])
        df = df.dropna(subset=["label"])
        df["subject"] = subject
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
        df = df.dropna(subset=["timestamp"])
        df = df.sort_values("timestamp").reset_index(drop=True)
        df["relative_time"] = df["timestamp"] - df["timestamp"].iloc[0]
        frames.append(df)
    if not frames:
        return pd.DataFrame(), missing
    return pd.concat(frames, ignore_index=True), missing

def assign_windows(df: pd.DataFrame, window_size: float = WINDOW_SIZE, overlap: float = OVERLAP) -> pd.DataFrame:
    if df.empty:
        return df
    stride = window_size - overlap
    if stride <= 0:
        raise ValueError("overlap must be smaller than window_size")
    df = df.copy()
    df["window_index"] = (
        df.groupby("subject")["relative_time"]
        .transform(lambda col: np.floor(col / stride))
        .astype(int)
    )
    df["window_start"] = df["window_index"] * stride
    df["window_end"] = df["window_start"] + window_size
    return df

def compute_label_counts(df: pd.DataFrame, by: str) -> pd.Series:
    if df.empty:
        return pd.Series(dtype=int)
    if by == "window":
        labels = df.groupby(["subject", "window_index"])["label"].first()
    elif by == "row":
        labels = df["label"]
    else:
        raise ValueError("by must be 'window' or 'row'")
    return labels.value_counts().sort_index()

## Load All Modality Data

Process all three modalities (OCOSense, eye tracker, Empatica):
1. Load raw CSV files for each subject
2. Segment data into 1-second windows
3. Compute label distributions at both window and row level
4. Track any missing files

This gives us a complete view of data availability across modalities.

In [ ]:
subjects = discover_subjects()
print(f"Discovered {len(subjects)} subject folders.")

segmented_data = {}
samples_per_window = {}
label_counts_window = {}
label_counts_row = {}
missing_files = {}

for idx, modality in enumerate(MODALITY_FOLDERS.keys(), 1):
    print(f"\n[{idx}/{len(MODALITY_FOLDERS)}] Processing modality: {modality}")
    df, missing = load_modality_data(modality, subjects)
    missing_files[modality] = missing
    df = assign_windows(df)
    segmented_data[modality] = df
    samples_per_window[modality] = df.groupby(["subject", "window_index"]).size() if not df.empty else pd.Series(dtype=int)
    label_counts_window[modality] = compute_label_counts(df, by="window")
    label_counts_row[modality] = compute_label_counts(df, by="row")
    if df.empty:
        print(f"  → {modality}: no data found")
    else:
        subjects_present = df['subject'].nunique()
        print(f"  → {modality}: {len(df):,} rows across {subjects_present} subjects")
        if missing:
            print(f"    Missing files: {len(missing)}")

## Label Distribution per Window

Visualize how many 1-second windows contain each label across the three modalities.

**What this shows:**
- Overall balance of experimental conditions in the dataset
- Consistency of label counts across modalities (bars should be similar heights)
- Any systematic differences in how labels are distributed

**Interpretation:**
- Similar bar heights across modalities = good synchronization
- Large differences might indicate timing issues or missing data

In [ ]:
all_labels = set()
for counts in label_counts_window.values():
    all_labels.update(counts.index)
label_order = ensure_label_order(all_labels)

if label_order:
    fig, ax = plt.subplots(figsize=(18, 10))
    bar_width = 0.25
    positions = np.arange(len(label_order))
    offset = (len(label_counts_window) - 1) * bar_width / 2
    max_height = 0
    for i, (modality, counts) in enumerate(label_counts_window.items()):
        counts = counts.reindex(label_order, fill_value=0)
        peak = counts.max() if not counts.empty else 0
        max_height = max(max_height, peak)
        pos = positions - offset + i * bar_width
        bars = ax.bar(pos, counts.values, bar_width, label=modality.title())
        for x, y in zip(pos, counts.values):
            if y > 0:
                ax.text(x, y + max(1, max_height * 0.02), f"{int(y)}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(positions)
    ax.set_xticklabels(label_order, rotation=45, ha="right", fontsize=11)
    ax.set_ylabel(f"Count per {WINDOW_SIZE:.0f}s window", fontsize=12)
    ax.set_title("Label distribution per window by modality", fontsize=14, pad=20)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis='y', labelsize=11)
    plt.tight_layout()
else:
    print("No labels available for plotting.")

### Window-Level Statistics

Print detailed statistics about label distribution per window for each modality.

In [ ]:
for modality, counts in label_counts_window.items():
    print(modality.title())
    if counts.empty:
        print("  No labelled windows available.")
        continue
    total = counts.sum()
    print(f"  Total windows: {total}")
    print(f"  Labels present: {len(counts[counts > 0])}")
    top = counts.sort_values(ascending=False).head(5)
    for label, value in top.items():
        pct = (value / total) * 100 if total else 0
        print(f"    {label}: {value} ({pct:.1f}%)")
    print()

## Label Distribution by Row Count

Visualize the raw sample counts for each label across modalities.

**What this shows:**
- Total number of data samples (rows) for each experimental condition
- Different modalities have different sampling rates, so row counts will differ
- OCOSense: ~50 Hz, eye tracker: ~60-90 Hz, Empatica: varies by sensor

**Why this matters:**
- Helps verify that all modalities captured data for all conditions
- Shows relative data volume available for each task type
- Useful for understanding statistical power per condition

In [ ]:
all_row_labels = set()
for counts in label_counts_row.values():
    all_row_labels.update(counts.index)
row_label_order = ensure_label_order(all_row_labels)

if row_label_order:
    fig, ax = plt.subplots(figsize=(18, 10))
    bar_width = 0.25
    positions = np.arange(len(row_label_order))
    offset = (len(label_counts_row) - 1) * bar_width / 2
    max_height = 0
    for i, (modality, counts) in enumerate(label_counts_row.items()):
        counts = counts.reindex(row_label_order, fill_value=0)
        peak = counts.max() if not counts.empty else 0
        max_height = max(max_height, peak)
        pos = positions - offset + i * bar_width
        bars = ax.bar(pos, counts.values, bar_width, label=modality.title())
        for x, y in zip(pos, counts.values):
            if y > 0:
                ax.text(x, y + max(1, max_height * 0.02), f"{int(y)}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(positions)
    ax.set_xticklabels(row_label_order, rotation=45, ha="right", fontsize=11)
    ax.set_ylabel("Row count", fontsize=12)
    ax.set_title("Label distribution by row across modalities", fontsize=14, pad=20)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis='y', labelsize=11)
    plt.tight_layout()
else:
    print("No row-level labels available for plotting.")

### Row-Level Statistics

Print detailed statistics about raw sample counts for each modality.

In [ ]:
for modality, counts in label_counts_row.items():
    print(modality.title())
    if counts.empty:
        print("  No labelled rows available.")
        continue
    total = counts.sum()
    print(f"  Total rows: {total}")
    print(f"  Labels present: {len(counts[counts > 0])}")
    top = counts.sort_values(ascending=False).head(5)
    for label, value in top.items():
        pct = (value / total) * 100 if total else 0
        print(f"    {label}: {value} ({pct:.1f}%)")
    print()

### Missing Files Check

Verify that all expected data files are present for each modality.

In [ ]:
any_missing = False
for modality, paths in missing_files.items():
    if not paths:
        continue
    any_missing = True
    print(f"{modality.title()} missing {len(paths)} file(s):")
    for path in paths[:5]:
        print(f"  - {path}")
    if len(paths) > 5:
        print("  ...")

if not any_missing:
    print("All modality files found for discovered subjects.")

## Cross-Modality Synchronization Analysis

The following sections analyze how well the modalities are synchronized by comparing:
1. Temporal alignment of labels
2. Label agreement across modalities
3. Per-subject synchronization quality

## Compute Label Agreement Across Modalities

This function compares labels across all three modalities (OCOSense, eye tracker, Empatica) at each time window.

**How it works:**
1. For each subject, align data from all modalities by window index
2. Find windows that exist in all three modalities (common windows)
3. Check if all modalities agree on the label for each window
4. Track disagreements and their patterns

**Output:**
A DataFrame containing:
- `all_agree`: Boolean indicating if all 3 modalities have the same label
- `num_unique_labels`: How many different labels exist for that window (1, 2, or 3)
- Individual label columns for each modality (ocosense_label, eyetracker_label, empatica_label)

This is the foundation for all synchronization quality metrics.

In [ ]:
def compute_label_agreement(data_dict, subjects, window_size=WINDOW_SIZE):
    """
    Compute label agreement between modalities at each time window.
    Returns a DataFrame with agreement metrics per subject and overall.
    """
    agreement_results = []
    
    for subject in tqdm(subjects, desc="Computing label agreement"):
        # Get data for each modality for this subject
        modality_windows = {}
        for modality, df in data_dict.items():
            subject_df = df[df['subject'] == subject].copy()
            if subject_df.empty:
                continue
            # Get label per window
            window_labels = subject_df.groupby('window_index')['label'].first()
            modality_windows[modality] = window_labels
        
        if len(modality_windows) < 2:
            continue
            
        # Find common windows across all modalities
        common_indices = set.intersection(*[set(labels.index) for labels in modality_windows.values()])
        
        if not common_indices:
            continue
            
        for window_idx in common_indices:
            labels = {mod: modality_windows[mod].loc[window_idx] for mod in modality_windows.keys()}
            
            # Check if all modalities agree
            unique_labels = set(labels.values())
            all_agree = len(unique_labels) == 1
            
            agreement_results.append({
                'subject': subject,
                'window_index': window_idx,
                'all_agree': all_agree,
                'num_unique_labels': len(unique_labels),
                **{f'{mod}_label': label for mod, label in labels.items()}
            })
    
    return pd.DataFrame(agreement_results)

agreement_df = compute_label_agreement(segmented_data, subjects)
print(f"\nAnalyzed {len(agreement_df)} common windows across modalities")

### Overall and Per-Subject Agreement Statistics

Print comprehensive synchronization metrics:

**Overall Metrics:**
- **Perfect agreement rate**: % of windows where all 3 modalities have identical labels
- **Disagreement breakdown**: How many windows have 2 vs 3 different labels

**Per-Subject Metrics:**
- Agreement rate for each subject
- Identifies best and worst synchronized subjects
- Helps pinpoint potential data quality issues

**What to look for:**
- Overall agreement >95% is excellent
- 90-95% is good
- <90% suggests synchronization issues that need investigation

In [ ]:
# Overall agreement statistics
if not agreement_df.empty:
    overall_agreement = agreement_df['all_agree'].mean() * 100
    print(f"\n{'='*60}")
    print(f"OVERALL SYNCHRONIZATION METRICS")
    print(f"{'='*60}")
    print(f"Total windows analyzed: {len(agreement_df):,}")
    print(f"Perfect agreement (all modalities): {overall_agreement:.2f}%")
    print(f"Windows with 2 unique labels: {(agreement_df['num_unique_labels'] == 2).sum():,} ({(agreement_df['num_unique_labels'] == 2).mean() * 100:.2f}%)")
    print(f"Windows with 3 unique labels: {(agreement_df['num_unique_labels'] == 3).sum():,} ({(agreement_df['num_unique_labels'] == 3).mean() * 100:.2f}%)")
    
    # Per-subject agreement
    print(f"\n{'='*60}")
    print(f"PER-SUBJECT AGREEMENT")
    print(f"{'='*60}")
    subject_agreement = agreement_df.groupby('subject')['all_agree'].agg(['mean', 'count'])
    subject_agreement.columns = ['agreement_rate', 'num_windows']
    subject_agreement['agreement_pct'] = subject_agreement['agreement_rate'] * 100
    subject_agreement = subject_agreement.sort_values('agreement_pct', ascending=False)
    
    print(f"\nTop 5 subjects (best synchronization):")
    for idx, (subject, row) in enumerate(subject_agreement.head(5).iterrows(), 1):
        print(f"  {idx}. Subject {subject:02d}: {row['agreement_pct']:.2f}% ({int(row['num_windows'])} windows)")
    
    print(f"\nBottom 5 subjects (worst synchronization):")
    for idx, (subject, row) in enumerate(subject_agreement.tail(5).iterrows(), 1):
        print(f"  {idx}. Subject {subject:02d}: {row['agreement_pct']:.2f}% ({int(row['num_windows'])} windows)")
else:
    print("No common windows found for agreement analysis.")

### Visualize Per-Subject Synchronization Quality

Two complementary visualizations:

**Left Plot - Per-Subject Agreement:**
- Each bar represents one subject's synchronization quality
- Color coding: Green (>90%), Orange (80-90%), Red (<80%)
- Quickly identify problematic subjects that may need data review

**Right Plot - Disagreement Types:**
- Shows distribution of how modalities disagree
- "2 different labels" = two modalities agree, one disagrees
- "3 different labels" = all three modalities have different labels (worst case)

**Use case:**
- Identify subjects to exclude due to poor synchronization
- Understand whether disagreements are systematic or random

In [ ]:
# Visualize per-subject agreement rates
if not agreement_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    
    # Plot 1: Per-subject agreement rate
    subject_agreement_sorted = subject_agreement.sort_values('subject')
    colors = ['green' if x > 90 else 'orange' if x > 80 else 'red' for x in subject_agreement_sorted['agreement_pct']]
    
    bars = ax1.bar(range(len(subject_agreement_sorted)), 
                   subject_agreement_sorted['agreement_pct'],
                   color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax1.set_xlabel('Subject', fontsize=12)
    ax1.set_ylabel('Agreement Rate (%)', fontsize=12)
    ax1.set_title('Cross-Modality Label Agreement by Subject', fontsize=14, pad=15)
    ax1.set_xticks(range(len(subject_agreement_sorted)))
    ax1.set_xticklabels([f"{int(s)}" for s in subject_agreement_sorted.index], rotation=45, fontsize=10)
    ax1.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='90% threshold')
    ax1.axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='80% threshold')
    ax1.grid(axis='y', alpha=0.3)
    ax1.legend(fontsize=10)
    ax1.tick_params(axis='y', labelsize=11)
    
    # Plot 2: Distribution of disagreement types
    disagreement_df = agreement_df[~agreement_df['all_agree']]
    if not disagreement_df.empty:
        disagreement_counts = disagreement_df['num_unique_labels'].value_counts().sort_index()
        ax2.bar(disagreement_counts.index, disagreement_counts.values, 
                color='coral', alpha=0.7, edgecolor='black', linewidth=0.5)
        ax2.set_xlabel('Number of Different Labels', fontsize=12)
        ax2.set_ylabel('Count of Windows', fontsize=12)
        ax2.set_title('Distribution of Label Disagreements', fontsize=14, pad=15)
        ax2.set_xticks(disagreement_counts.index)
        ax2.grid(axis='y', alpha=0.3)
        ax2.tick_params(labelsize=11)
        
        # Add percentage annotations
        total_disagreements = len(disagreement_df)
        for idx, count in disagreement_counts.items():
            pct = (count / total_disagreements) * 100
            ax2.text(idx, count, f'{count}\n({pct:.1f}%)', 
                    ha='center', va='bottom', fontsize=10)
    else:
        ax2.text(0.5, 0.5, 'No disagreements found!', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        ax2.set_title('Distribution of Label Disagreements', fontsize=14, pad=15)
    
    plt.tight_layout()
    plt.show()
else:
    print("No agreement data available for visualization.")

### Pairwise Modality Agreement Analysis

Compare synchronization between each pair of modalities:
- **OCOSense vs eye tracker**: How well do the smart-glasses and gaze streams align?
- **OCOSense vs Empatica**: How well do the smart-glasses and physiological streams align?
- **Eye tracker vs Empatica**: How well do gaze and physiology align?

**Why pairwise matters:**
- Identifies which specific modality pairs have synchronization issues
- A low pairwise score indicates timing problems between those two modalities
- All pairwise scores should be similar to the overall agreement rate

**Example interpretation:**
- If OCOSense-eye tracker = 98% but OCOSense-Empatica = 85%, Empatica may have timing drift
- If all pairs are high (>95%), synchronization is excellent across the board

In [ ]:
# Pairwise modality agreement analysis
if not agreement_df.empty and len(MODALITY_FOLDERS) > 1:
    print(f"\n{'='*60}")
    print(f"PAIRWISE MODALITY AGREEMENT")
    print(f"{'='*60}\n")
    
    modality_pairs = []
    modalities = list(MODALITY_FOLDERS.keys())
    
    for i in range(len(modalities)):
        for j in range(i+1, len(modalities)):
            mod1, mod2 = modalities[i], modalities[j]
            if f'{mod1}_label' in agreement_df.columns and f'{mod2}_label' in agreement_df.columns:
                pair_agreement = (agreement_df[f'{mod1}_label'] == agreement_df[f'{mod2}_label']).mean() * 100
                modality_pairs.append({
                    'pair': f"{mod1.title()} vs {mod2.title()}",
                    'agreement': pair_agreement
                })
                print(f"{mod1.title():12} vs {mod2.title():12}: {pair_agreement:.2f}% agreement")
    
    # Visualize pairwise agreement
    if modality_pairs:
        fig, ax = plt.subplots(figsize=(10, 6))
        pairs_df = pd.DataFrame(modality_pairs)
        colors = ['green' if x > 90 else 'orange' if x > 80 else 'red' for x in pairs_df['agreement']]
        
        bars = ax.barh(pairs_df['pair'], pairs_df['agreement'], 
                       color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Agreement Rate (%)', fontsize=12)
        ax.set_title('Pairwise Label Agreement Between Modalities', fontsize=14, pad=15)
        ax.axvline(x=90, color='green', linestyle='--', alpha=0.5, label='90% threshold')
        ax.axvline(x=80, color='orange', linestyle='--', alpha=0.5, label='80% threshold')
        ax.grid(axis='x', alpha=0.3)
        ax.legend(fontsize=10)
        ax.tick_params(labelsize=11)
        
        # Add value labels
        for bar, val in zip(bars, pairs_df['agreement']):
            ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, 
                   f'{val:.1f}%', va='center', fontsize=11)
        
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for pairwise analysis.")

### Temporal Alignment Visualization

Visual inspection of label sequences over time for one sample subject.

**What this shows:**
- Each subplot shows one modality's label sequence
- Time on X-axis, labels on Y-axis
- All three modalities should show the same step pattern if perfectly synchronized

**How to interpret:**
- **Perfect sync**: All three plots show identical patterns (steps at same times)
- **Timing drift**: Patterns are similar but shifted horizontally
- **Label errors**: Different labels at the same time points
- **Missing data**: Gaps in one modality but not others

**Use case:**
- Visual QA to spot systematic timing issues
- Identify whether problems are gradual drift or sudden jumps
- Verify that phase transitions happen simultaneously across modalities

In [ ]:
# Temporal alignment visualization for a sample subject
def plot_temporal_alignment(subject_id, data_dict, max_windows=100):
    """
    Plot label sequences across modalities for visual inspection of synchronization.
    """
    fig, axes = plt.subplots(len(data_dict), 1, figsize=(20, 3*len(data_dict)), sharex=True)
    if len(data_dict) == 1:
        axes = [axes]
    
    label_to_num = {label: idx for idx, label in enumerate(BASE_LABEL_ORDER)}
    
    for ax_idx, (modality, df) in enumerate(data_dict.items()):
        subject_df = df[df['subject'] == subject_id].copy()
        if subject_df.empty:
            axes[ax_idx].text(0.5, 0.5, f'No data for {modality}', 
                            ha='center', va='center', transform=axes[ax_idx].transAxes, fontsize=12)
            axes[ax_idx].set_ylabel(modality.title(), fontsize=12)
            continue
        
        # Get window labels
        window_labels = subject_df.groupby('window_index').agg({
            'label': 'first',
            'window_start': 'first'
        }).head(max_windows)
        
        # Convert labels to numbers for plotting
        window_labels['label_num'] = window_labels['label'].map(label_to_num)
        
        # Plot as step function
        axes[ax_idx].step(window_labels['window_start'], window_labels['label_num'], 
                         where='post', linewidth=2, color=f'C{ax_idx}')
        axes[ax_idx].set_ylabel(modality.title(), fontsize=12)
        axes[ax_idx].set_yticks(range(len(BASE_LABEL_ORDER)))
        axes[ax_idx].set_yticklabels(BASE_LABEL_ORDER, fontsize=9)
        axes[ax_idx].grid(True, alpha=0.3)
        axes[ax_idx].tick_params(axis='x', labelsize=11)
        
    axes[-1].set_xlabel('Time (seconds)', fontsize=12)
    fig.suptitle(f'Temporal Label Alignment for Subject {subject_id} (first {max_windows} windows)', 
                fontsize=14, y=0.995)
    plt.tight_layout()
    plt.show()

# Plot for a sample subject
if subjects and segmented_data:
    sample_subject = subjects[0]  # Use first subject as example
    print(f"Visualizing temporal alignment for Subject {sample_subject}")
    plot_temporal_alignment(sample_subject, segmented_data, max_windows=150)

### Common Disagreement Pattern Analysis

Identify the most frequent label disagreement patterns across modalities.

**What this shows:**
- Top 10 specific combinations where modalities disagree
- Format: `modality:label | modality:label | modality:label`
- Frequency and percentage of each pattern

**Why this matters:**
- **Systematic patterns** suggest specific synchronization bugs
  - E.g., "ocosense:task_A | eyetracker:task_A | empatica:questionnaire" repeated many times
  - May indicate Empatica started/stopped recording at different times
- **Random patterns** suggest noise or boundary effects
  - E.g., disagreements at task transitions due to timing precision
- **Common label pairs** help prioritize fixes
  - If "questionnaire vs rest" appears often, check rest period timing

**Action items:**
- Frequent patterns involving specific labels → review those experimental phases
- One modality consistently wrong → check that modality's timestamp synchronization
- Patterns at specific subjects → review those recording sessions

In [ ]:
# Analyze common disagreement patterns
if not agreement_df.empty:
    disagreements = agreement_df[~agreement_df['all_agree']].copy()
    
    if not disagreements.empty:
        print(f"\n{'='*60}")
        print(f"COMMON DISAGREEMENT PATTERNS")
        print(f"{'='*60}\n")
        
        # Find most common label combinations in disagreements
        modalities = list(MODALITY_FOLDERS.keys())
        label_cols = [f'{mod}_label' for mod in modalities if f'{mod}_label' in disagreements.columns]
        
        if len(label_cols) >= 2:
            # Create a combined key for disagreement patterns
            disagreements['pattern'] = disagreements[label_cols].apply(
                lambda row: ' | '.join([f"{col.split('_')[0]}:{row[col]}" for col in label_cols]), 
                axis=1
            )
            
            pattern_counts = disagreements['pattern'].value_counts().head(10)
            
            print(f"Top 10 most common disagreement patterns:")
            print(f"(Format: modality:label | modality:label | ...)\n")
            for idx, (pattern, count) in enumerate(pattern_counts.items(), 1):
                pct = (count / len(disagreements)) * 100
                print(f"{idx:2}. {pattern}")
                print(f"    Count: {count:,} ({pct:.2f}% of disagreements)\n")
            
            # Visualize top disagreement patterns
            fig, ax = plt.subplots(figsize=(14, 8))
            y_pos = range(len(pattern_counts))
            bars = ax.barh(y_pos, pattern_counts.values, color='indianred', alpha=0.7, 
                          edgecolor='black', linewidth=0.5)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([p[:80] + '...' if len(p) > 80 else p for p in pattern_counts.index], 
                              fontsize=10)
            ax.set_xlabel('Number of Windows', fontsize=12)
            ax.set_title('Top 10 Most Common Label Disagreement Patterns', fontsize=14, pad=15)
            ax.grid(axis='x', alpha=0.3)
            ax.tick_params(axis='x', labelsize=11)
            
            # Add count labels
            for bar, count in zip(bars, pattern_counts.values):
                ax.text(count + max(pattern_counts.values) * 0.01, 
                       bar.get_y() + bar.get_height()/2,
                       f'{count:,}', va='center', fontsize=10)
            
            plt.tight_layout()
            plt.show()
        else:
            print("Not enough modality data for pattern analysis.")
    else:
        print("\nNo disagreements found - perfect synchronization across all modalities!")
else:
    print("No agreement data available for disagreement analysis.")